# Task 1: Meta-Software Development

This notebook demonstrates an AI-assisted software generation workflow for the AI Quote Generator project. It uses a Large Language Model API to generate application code, frontend code, SDLC documentation, and a UML sequence diagram.

**Generated artefacts:**
- `app.py`: Flask backend API
- `templates/index.html`: Frontend web page
- `docs/sdlc_requirements.md`: Traditional SDLC requirements documentation
- `diagram.puml`: PlantUML sequence diagram


## Cell 1: Environment Initialisation

This cell imports the required Python libraries, disables insecure request warnings for the coursework API environment, clears proxy variables, and prepares the target project folder structure.

In [ ]:
import os
import requests
import json
import urllib3

# Suppress warnings caused by verify=False in the coursework API request.
urllib3.disable_warnings()

# Clear proxy settings to avoid network routing issues in the coursework environment.
os.environ["http_proxy"] = ""
os.environ["https_proxy"] = ""
os.environ["all_proxy"] = ""

project_dir = "AI_Quote_Generator"
if os.path.basename(os.getcwd()) == project_dir:
    project_dir = "."

templates_dir = os.path.join(project_dir, "templates")
static_dir = os.path.join(project_dir, "static")
docs_dir = os.path.join(project_dir, "docs")

os.makedirs(templates_dir, exist_ok=True)
os.makedirs(static_dir, exist_ok=True)
os.makedirs(docs_dir, exist_ok=True)

print("Folder structure initialized successfully")


## Cell 2: Core LLM Calling Function

This cell defines the reusable `ask_ai(prompt)` function. The function sends prompts to the LLM API and returns clean text output suitable for writing directly into project files.

**Security note:** the submitted notebook uses `YOUR_API_KEY_HERE` as a placeholder. A real key should only be inserted locally for testing and must not be committed to GitHub.

In [ ]:
API_KEY = "YOUR_API_KEY_HERE"
API_URL = "https://api.apifree.ai/v1/chat/completions"
MODEL_NAME = "deepseek-ai/deepseek-v3.2"

def ask_ai(prompt):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
        "User-Agent": "Mozilla/5.0"
    }

    data = {
        "max_tokens": 8192,
        "messages": [
            {"content": prompt, "role": "user"}
        ],
        "model": MODEL_NAME,
        "stream": False,
        "temperature": 1,
        "top_p": 1
    }

    try:
        response = requests.post(
            API_URL,
            headers=headers,
            json=data,
            verify=False,
            proxies={"http": "", "https": ""},
            timeout=60
        )

        response.raise_for_status()
        result = response.json()
        text = result["choices"][0]["message"]["content"]

        # Remove common Markdown code-fence wrappers so the output can be saved as clean files.
        text = text.replace("```python", "")
        text = text.replace("```html", "")
        text = text.replace("```css", "")
        text = text.replace("```javascript", "")
        text = text.replace("```js", "")
        text = text.replace("```plantuml", "")
        text = text.replace("```markdown", "")
        text = text.replace("```", "")

        return text.strip()

    except Exception as e:
        print("Request failed:", e)
        return None


## Cell 3: Automated Software Artefact Generation

This cell performs the actual meta-software development process. It prompts the LLM to generate the backend, frontend, traditional SDLC requirements documentation, and UML diagram, then writes each generated artefact into the correct project location.

In [ ]:
def write_generated_file(relative_path, content):
    output_path = os.path.join(project_dir, relative_path)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as file:
        file.write(content)


generation_tasks = [
    {
        "label": "Flask backend",
        "path": "app.py",
        "prompt": "You are a senior Python engineer. Write a Flask backend app.py. It must include a root route '/' that renders index.html, a '/api/quote' route that returns a random inspirational quote as JSON, and a '/generated-image' route that retrieves an automatically generated image from Pollinations API. Return only pure Python code without explanation."
    },
    {
        "label": "Frontend website",
        "path": os.path.join("templates", "index.html"),
        "prompt": "Write a frontend index.html. It should display an automatically generated image from Pollinations API or from the Flask '/generated-image' route, show a quote text below it, and include a button. When the button is clicked, JavaScript should fetch '/api/quote' and update the quote text. Return only pure HTML code without explanation."
    },
    {
        "label": "SDLC requirements document",
        "path": os.path.join("docs", "sdlc_requirements.md"),
        "prompt": "Write a concise Software Requirements Specification document in Markdown for an AI Quote Generator web application. Include: purpose, scope, stakeholders, functional requirements, non-functional requirements, user stories, acceptance criteria, system architecture, deployment assumptions, and risks. Return Markdown only."
    },
    {
        "label": "UML sequence diagram",
        "path": "diagram.puml",
        "prompt": "Write a PlantUML sequence diagram in English. Describe this process: user opens the web page, frontend requests the Flask backend, backend returns the page, frontend displays an automatically generated image, user clicks the quote button, frontend requests /api/quote, Flask backend returns JSON data, and frontend updates the quote text. Return only the content from @startuml to @enduml."
    }
]


for task in generation_tasks:
    print(f"Generating {task['label']} ...")
    generated_text = ask_ai(task["prompt"])

    if generated_text is None:
        print(f"{task['label']} generation failed. Please check API_KEY, network connection, or API_URL.")
        break

    write_generated_file(task["path"], generated_text)
    print(f"Saved {task['path']}")
else:
    print("All software artefacts generated successfully")
